# Sicilian–English–Italian Neural Machine Translation
### A reproducible, end-to-end pipeline

This notebook reproduces our whole model from start to finish, structured like a short
paper. Each section explains *what* and *why* (as in the slides), then runs the code — the
implementations live in the repo's `.py` files (`experiments/nllb/nllb_pipeline.py`,
`experiments/dataset/normalize_scn.py`), the notebook only orchestrates.

**Runtime → Change runtime type → GPU.**

## 1. Introduction

Sicilian is a low-resource Romance language, absent from the major MT services. Wdowiak's
*Tradutturi Sicilianu* is the published state of the art, built by training a model from
scratch on tens of millions of pairs (his "reverse-training" recipe).

**Research question.** Can a *pretrained* massively-multilingual model (NLLB-200), adapted
with parameter-efficient fine-tuning, reach that quality? And where it does not, is the gap
due to **method** or to **data**?

**Approach.** We (i) assemble a reproducible corpus, (ii) normalize to Standard Sicilian,
(iii) fine-tune NLLB-200 with LoRA, (iv) evaluate on a frozen literary test set, and
(v) apply the feasible parts of his recipe (multilingual, back-translation) plus a
normalization ablation, to attribute the residual gap to method vs data.

## 2. Setup
Clone the repo for the `.py` code, install dependencies, mount Drive.

In [1]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
sys.path += ['SicilianNMT/experiments/nllb', 'SicilianNMT/experiments/dataset']
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'; DATA = f'{OUT}/data'
def read(p): return open(p, encoding='utf-8').read().splitlines()
from nllb_pipeline import load_base, attach_lora, build_dataset, finetune, translate, score
from normalize_scn import normalize
print('ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.5 MB/s eta 0:00:00
Mounted at /content/drive
ready


## 3. Data

### 3.1 Sources, extraction & alignment
The parallel text is built automatically: extract the *Arba Sicula* PDFs with PyMuPDF,
classify each page's language, confirm facing pairs and align sentences with **LaBSE**
embeddings + a monotonic dynamic program (no dictionary, no hunalign). We then add the
curated Napizia NLLB / WikiMatrix datasets and deduplicate. Code:
`experiments/extraction/build_all.py` and `experiments/dataset/assemble.py` (they require the
PDFs and the public datasets). A 1k/1k valid/test split is **frozen** so every result stays
comparable.

### 3.2 Loading the prepared dataset

In [2]:
train = {l: read(f'{DATA}/train.{l}') for l in ('scn', 'en')}
test  = {l: read(f'{DATA}/test.{l}')  for l in ('scn', 'en')}
print('train', len(train['scn']), '| test', len(test['scn']))
print('sample pair:\n  scn:', train['scn'][0], '\n  en :', train['en'][0])

train 27386 | test 1000
sample pair:
  scn: Esistunu na para di libbra di grammatica ma sunnu opiri di consultazioni. 
  en : There are several grammars, but they are primarily reference books, not teach­ ing textbooks.


## 4. Preprocessing

### 4.1 Standardization (Standard Sicilian)
Sicilian has no single official orthography (many spellings, omitted accents). We normalize
toward **Standard Sicilian** — Cipolla's standard, augmented by Wdowiak for MT clarity — with
`normalize_scn.py`. The light **std** level applies spelling rules (*cchiù→chiù*,
*io/iu/eu→jo*, *non→nun*, *çiuri→ciuri*) and keeps accents/contractions; the **full** level
additionally uncontracts (*dû→di lu*) and ASCII-folds. We use **std** (its effect is measured
in the ablation, §7.3).

In [3]:
print('before:', test['scn'][0])
for d in (train, test):
    d['scn'] = [normalize(s, 'std') for s in d['scn']]
print('after :', test['scn'][0])

before: Nun eppimu mai lu piaciri di vidiri i so opiri ma sta pasta di sicuru fa arrisbigliari i papiddi gustativi da lingua di tutti, siciliani e no.
after : Nun eppimu mai lu piaciri di vidiri i so opiri ma sta pasta di sicuru fa arrisbigliari i papiddi gustativi da lingua di tutti, siciliani e no.


### 4.2 Tokenization & subwords
NLLB-200 carries its own SentencePiece tokenizer, so the model side needs no extra
tokenization. For the from-scratch Sockeye baseline we instead port the Napizia tokenizer
(ASCII-fold + uncontract) and bias a small joint BPE toward textbook **desinences**
(*Carinisi are dogs!* → `car@@ in@@ isi are dogs !`), cutting the vocabulary 14k→2k — see
`experiments/tokenization/` and `experiments/baseline/`.

## 5. Model & training

### 5.1 NLLB-200 + LoRA
NLLB-200 is a massively multilingual encoder–decoder that **supports Sicilian** (`scn_Latn`) —
the equivalent of the costly pretraining a from-scratch system would need. We adapt it with
**LoRA**: freeze the weights $W_0$ and learn a low-rank update $W=W_0+\frac{\alpha}{r}BA$
($r{=}32$) on the attention projections — ~1.3% of the parameters, fits a T4.

In [ ]:
model, tok = load_base('facebook/nllb-200-1.3B')
ft = attach_lora(model)

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 18,874,368 || all params: 1,389,512,704 || trainable%: 1.3583


### 5.2 Bidirectional fine-tune (scn↔en)
We train **both directions at once** so one adapter serves both and the weak reverse
direction is rescued (each example is tagged with its forced target language). ~hours on a T4.

In [ ]:
ds = build_dataset(tok, [(train['scn'], train['en'], 'scn', 'en'),
                         (train['en'], train['scn'], 'en', 'scn')])
ft = finetune(ft, tok, ds, out_dir=f'{OUT}/nllb-lora-bidir-1.3B', epochs=2)

Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Step,Training Loss
100,8.645858
200,8.172184
300,8.127043
400,7.859276
500,7.751219
600,7.569832
700,7.929872
800,7.587246
900,7.590767
1000,7.599669


## 6. Evaluation (test)

### 6.1 Metrics
We score on the **frozen** literary test set with **sacreBLEU** (BLEU, `tok:13a`) and
**chrF** — chrF being robust for a morphology-rich language. We report both directions.

### 6.2 Main result

In [ ]:
results = {}
for s, t in [('scn', 'en'), ('en', 'scn')]:
    results[f'{s}->{t}'] = score(translate(ft, tok, test[s], s, t), test[t])
    print(f'{s}->{t}:  BLEU={results[f"{s}->{t}"][0]}  chrF={results[f"{s}->{t}"][1]}')
json.dump(results, open(f'{OUT}/results_reproduce.json', 'w'), indent=2)

scn->en:  BLEU=31.21  chrF=56.77
en->scn:  BLEU=19.03  chrF=49.91


## 7. Further experiments
We now apply the feasible parts of Wdowiak's recipe on top of NLLB, plus a controlled
ablation. Each is independent and **reloads a fresh model** (heavy; run what you need). The
main result above is already recorded, so these do not affect it.

### 7.1 Italian bridge — one trilingual model
Fine-tune four directions jointly (scn↔en + it↔scn) into a single adapter. Needs
`itsc_{train,test}.{it,scn}` on Drive (frozen WikiMatrix split).

In [ ]:
isc = {s: {l: read(f'{DATA}/itsc_{s}.{l}') for l in ('it', 'scn')} for s in ('train', 'test')}
for s in ('train', 'test'):
    isc[s]['scn'] = [normalize(x, 'std') for x in isc[s]['scn']]
m, tok = load_base('facebook/nllb-200-1.3B'); mft = attach_lora(m)
mft = finetune(mft, tok, build_dataset(tok, [
    (train['scn'], train['en'], 'scn', 'en'), (train['en'], train['scn'], 'en', 'scn'),
    (isc['train']['it'], isc['train']['scn'], 'it', 'scn'),
    (isc['train']['scn'], isc['train']['it'], 'scn', 'it')]),
    out_dir=f'{OUT}/nllb-lora-multi-1.3B', epochs=1)
for s, t, te in [('scn','en',test), ('en','scn',test),
                 ('it','scn',isc['test']), ('scn','it',isc['test'])]:
    print(f'{s}->{t}:', score(translate(mft, tok, te[s], s, t), te[t]))

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

trainable params: 18,874,368 || all params: 1,389,512,704 || trainable%: 1.3583


Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Map:   0%|          | 0/9389 [00:00<?, ? examples/s]

Map:   0%|          | 0/9389 [00:00<?, ? examples/s]

Step,Training Loss
100,8.900496
200,8.081706
300,8.232970
400,8.216196
500,7.908330
600,7.747371
700,7.812760
800,7.713580
900,7.780746
1000,8.073116


scn->en: (30.97, 56.63)
en->scn: (17.85, 48.82)
it->scn: (27.48, 51.66)
scn->it: (43.7, 59.89)


### 7.2 Back-translation
Back-translate in-domain monolingual English (`mono_en.txt`, harvested from unaligned text)
en→scn with the bidirectional adapter to make synthetic Sicilian, then fine-tune scn→en on
real + synthetic. The English targets stay real, so the decoder learns better English.

In [ ]:
mono = read(f'{DATA}/mono_en.txt')[:15000]
m, tok = load_base('facebook/nllb-200-1.3B', adapter=f'{OUT}/nllb-lora-bidir-1.3B')
synth = translate(m, tok, mono, 'en', 'scn', bs=16, beams=4)
syn, seen = [], set(train['scn'])
for sc, en in zip(synth, mono):
    sc, en = sc.strip(), en.strip()
    if sc and en and sc not in seen and 0.5 <= len(sc.split())/max(1, len(en.split())) <= 2.0:
        seen.add(sc); syn.append((sc, en))
print(f'kept {len(syn)} synthetic pairs')
bft = attach_lora(m)
bft = finetune(bft, tok, build_dataset(tok, [(train['scn'] + [s for s, _ in syn],
               train['en'] + [e for _, e in syn], 'scn', 'en')]),
               out_dir=f'{OUT}/nllb-lora-bt-1.3B', epochs=1)
print('BT scn->en:', score(translate(bft, tok, test['scn'], 'scn', 'en'), test['en']))

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

kept 7498 synthetic pairs
trainable params: 18,874,368 || all params: 1,389,512,704 || trainable%: 1.3583


Map:   0%|          | 0/34884 [00:00<?, ? examples/s]

Step,Training Loss
100,5.755824
200,5.707099
300,5.664911
400,5.797744
500,5.713551
600,5.687564
700,5.598663
800,5.643572
900,5.796688
1000,5.548970


BT scn->en: (31.66, 57.25)


### 7.3 Ablation — does normalization help NLLB?
A controlled comparison: three arms (**raw / std / full**), scn→en only, a fresh model each,
**identical English references** (so BLEU is directly comparable). Variants are generated
in-notebook by `normalize_scn`.

In [ ]:
raw = {s: read(f'{DATA}/{s}.scn') for s in ('train', 'test')}   # un-normalized
abl = {}
for arm in ('raw', 'std', 'full'):
    f = (lambda x: x) if arm == 'raw' else (lambda x, a=arm: normalize(x, a))
    m, tok = load_base('facebook/nllb-200-1.3B'); aft = attach_lora(m)
    aft = finetune(aft, tok, build_dataset(tok, [([f(x) for x in raw['train']], train['en'], 'scn', 'en')]),
                   out_dir=f'{OUT}/abl-{arm}', epochs=1)
    abl[arm] = score(translate(aft, tok, [f(x) for x in raw['test']], 'scn', 'en'), test['en'])
    print(f'{arm}:', abl[arm])
print(abl)

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

### 7.4 The two missing directions — it↔en on FLORES-200
Wdowiak reports all **six** directions among {scn, en, it}; our per-pair test sets only cover
four. FLORES-200 devtest (1012 sentences, multi-parallel, includes `scn_Latn`) fills the gap:
we score it↔en — directions the trilingual adapter **never saw** in fine-tuning — for both the
zero-shot base and the adapter, which doubles as a catastrophic-forgetting check. Set
`ALL_SIX = True` to score every direction on FLORES for one fully comparable table (slow).

In [ ]:
def flores(lang, split='devtest'):                      # multi-parallel, aligned by line
    p = f'flores200_dataset/{split}/{lang}.{split}'
    if not os.path.exists(p):
        os.system('wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz'
                  ' -O fl.tar.gz && tar -xzf fl.tar.gz')
    if os.path.exists(p):
        return read(p)
    from datasets import load_dataset                   # last resort: FLORES+ (gated, needs HF login)
    return [r['text'] for r in load_dataset('openlanguagedata/flores_plus', lang, split=split)]

from nllb_pipeline import LANG
fl = {l: flores(LANG[l]) for l in ('it', 'en', 'scn')}
ALL_SIX = False
pairs = ([(s, t) for s in ('scn', 'en', 'it') for t in ('scn', 'en', 'it') if s != t]
         if ALL_SIX else [('it', 'en'), ('en', 'it')])
flores_res, m = {}, None
for name, adapter in [('zero-shot', None), ('multi-ft', f'{OUT}/nllb-lora-multi-1.3B')]:
    m = None                                            # drop stale model BEFORE loading (T4 OOM)
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=adapter)
    for s, t in pairs:
        flores_res[f'{name} {s}->{t}'] = score(translate(m, tok, fl[s], s, t), fl[t])
        print(f'{name}  {s}->{t}:  BLEU={flores_res[f"{name} {s}->{t}"][0]}  chrF={flores_res[f"{name} {s}->{t}"][1]}')
json.dump(flores_res, open(f'{OUT}/results_flores.json', 'w'), indent=2)

### 7.5 Other languages to and from Sicilian
Does the trilingual adapter, fine-tuned only on scn↔en and it↔scn, already translate
*other* Romance languages to and from Sicilian? FLORES-200 also has Spanish, French and
Portuguese. This reuses `flores()` and the loader from §7.4 (run that cell first), and
tests zero-shot transfer through the shared Romance space.

In [ ]:
from nllb_pipeline import LANG
LANG.update({'es': 'spa_Latn', 'fr': 'fra_Latn', 'pt': 'por_Latn'})   # extra Romance probes
ol = {l: flores(LANG[l]) for l in ('es', 'fr', 'pt', 'scn')}
opairs = [('es','scn'),('fr','scn'),('pt','scn'),('scn','es'),('scn','fr'),('scn','pt')]
other_res = {}
m = None
m, tok = load_base('facebook/nllb-200-1.3B', adapter=f'{OUT}/nllb-lora-multi-1.3B')
for s, t in opairs:
    other_res[f'{s}->{t}'] = score(translate(m, tok, ol[s], s, t), ol[t])
    print(f'{s}->{t}:  BLEU={other_res[f"{s}->{t}"][0]}  chrF={other_res[f"{s}->{t}"][1]}')
json.dump(other_res, open(f'{OUT}/results_other_langs.json', 'w'), indent=2)

### 7.6 Combined fine-tune on the merged dataset (Colab Pro)
Stage-3 fine-tune on the merged **Eryk + ours** corpus (his curated Arba Sicula/Dieli/
Napizia/textbooks + our AS01-18/21 extraction + Napizia magazine, std-normalised;
assembled by `experiments/dataset/assemble_finetune.py`). The data is PRIVATE and lives on
**your** Drive at `OUT/finetune/` (never in the public repo). Use a **Colab Pro** A100/L4
(drop `batch_size` if a smaller GPU OOMs). Evaluated on the same frozen test as our other
results, normalised to the std space to match training, with a leakage guard.

In [ ]:
# --- stage-3 combined fine-tune (Colab Pro) ---
from normalize_scn import normalize
FT = f'{OUT}/finetune'
tr = {'scn': read(f'{FT}/train.scn-en.scn'), 'en': read(f'{FT}/train.scn-en.en')}
ti = {'scn': read(f'{FT}/train.scn-it.scn'), 'it': read(f'{FT}/train.scn-it.it')}
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')],
        'en': read(f'{DATA}/test.en')}
ban = set(test['scn']) | set(test['en'])                 # leakage guard vs frozen test
se = [(s, e) for s, e in zip(tr['scn'], tr['en']) if s not in ban and e not in ban]
print(f'scn-en {len(se)} (dropped {len(tr["scn"]) - len(se)} test-overlaps) | scn-it {len(ti["scn"])}')
m, tok = load_base('facebook/nllb-200-1.3B'); cft = attach_lora(m)
ds = build_dataset(tok, [([s for s, _ in se], [e for _, e in se], 'scn', 'en'),
                         ([e for _, e in se], [s for s, _ in se], 'en', 'scn'),
                         (ti['scn'], ti['it'], 'scn', 'it'),
                         (ti['it'], ti['scn'], 'it', 'scn')])
cft = finetune(cft, tok, ds, out_dir=f'{OUT}/nllb-lora-combined-1.3B',
               epochs=2, batch_size=16, grad_accum=1)   # Colab Pro A100/L4
res = {}
for s, t in [('scn', 'en'), ('en', 'scn')]:
    res[f'{s}->{t}'] = score(translate(cft, tok, test[s], s, t), test[t])
    print(f'{s}->{t}:  BLEU={res[f"{s}->{t}"][0]}  chrF={res[f"{s}->{t}"][1]}')
json.dump(res, open(f'{OUT}/results_combined.json', 'w'), indent=2)

### 7.7 Forward-scoring the back-translations (stage-2 selection, Colab Pro)
Eryk back-translated the NLLB Sicilian (treated as monolingual) into English/Italian; the
synthetic pairs must be **forward-scored** and filtered so we keep sentences whose Sicilian
is the literary standard (his reverse-training trick: standard Sicilian scores higher than
local dialect). His CPU `sockeye-score` would take ~9 months; this does it on GPU. We score
each `(source -> target)` pair by the model's length-normalised log-likelihood of the target
and keep the top fraction. **Run when Eryk's files land in `OUT/backtranslations/`;** confirm
with him whether to score with NLLB or his Sockeye checkpoint, and the keep-threshold.

In [ ]:
# --- forward-scoring scaffold (fill SRC_FILE/TGT_FILE/codes per direction) ---
import torch, torch.nn.functional as F, os
BT = f'{OUT}/backtranslations'                 # Eryk's back-translation files go here

@torch.no_grad()
def ll_scores(model, tok, src, tgt, src_code, tgt_code, bs=16, max_len=160):
    tok.src_lang, tok.tgt_lang = src_code, tgt_code
    dev = next(model.parameters()).device; out = []
    for i in range(0, len(src), bs):
        enc = tok(src[i:i+bs], return_tensors='pt', padding=True, truncation=True,
                  max_length=max_len).to(dev)
        lab = tok(text_target=tgt[i:i+bs], return_tensors='pt', padding=True,
                  truncation=True, max_length=max_len).to(dev)
        logits = model(**enc, labels=lab.input_ids).logits          # (B,T,V)
        lp = F.log_softmax(logits, -1).gather(-1, lab.input_ids.unsqueeze(-1)).squeeze(-1)
        mask = (lab.input_ids != tok.pad_token_id)
        out += ((lp * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()   # len-norm LL
    return out

def score_and_filter(src_file, tgt_file, src_code, tgt_code, keep_frac=0.5, adapter=None):
    src = read(f'{BT}/{src_file}'); tgt = read(f'{BT}/{tgt_file}')
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=adapter)
    sc = ll_scores(m, tok, src, tgt, src_code, tgt_code)
    order = sorted(range(len(sc)), key=lambda i: sc[i], reverse=True)   # higher = better
    keep = set(order[:int(len(order) * keep_frac)])
    base = tgt_file.rsplit('.', 1)[0]
    with open(f'{BT}/{base}.kept.src', 'w') as fs, open(f'{BT}/{base}.kept.tgt', 'w') as ft:
        for i in range(len(src)):
            if i in keep:
                fs.write(src[i] + '\n'); ft.write(tgt[i] + '\n')
    print(f'{src_file} -> {tgt_file}: kept {len(keep)}/{len(sc)} (top {keep_frac:.0%})')
    return sc

# Example (edit filenames/codes to match Eryk's files, then run per direction):
# LANG = {'scn':'scn_Latn','en':'eng_Latn','it':'ita_Latn'}
# score_and_filter('bt_en2scn.en', 'bt_en2scn.scn', 'eng_Latn', 'scn_Latn', keep_frac=0.5)
# score_and_filter('bt_it2scn.it', 'bt_it2scn.scn', 'ita_Latn', 'scn_Latn', keep_frac=0.5)
print('scaffold ready; place Eryk\'s files in', BT)

### 7.8 Full run: back-translation second-stage + combined fine-tune (Colab Pro)
The complete pipeline on our NLLB+LoRA stack, using Eryk's delivered data (all PRIVATE, on
`OUT/eryk/`). Stage 2: fine-tune NLLB-1.3B on the forward-scored back-translations (the K lowest-scoring
pairs per direction, `<unk>` lines, near-copies and the `<2xxx>` tag dropped) across scn/en/it + the natural WikiMatrix it-en. Stage 3:
continue on the high-quality combined set (`OUT/finetune/`), leakage-guarded, eval in std space.
**Long A100 run**; `TOPK` bounds it (raise once it works). NLLB replaces his Sockeye stages 1-2,
so the se36c/se37a checkpoints are not used here.

In [ ]:
import math
import re
from difflib import SequenceMatcher
from normalize_scn import normalize
BT = f'{OUT}/eryk'; FT = f'{OUT}/finetune'
TOPK = 100000                      # per-direction cap on back-translations (raise on A100)
COPY = 0.9                         # drop near-copies: the forward model scores copying as easy
MINW = 5                           # drop short web fragments (titles, menus), they score low too
_TAG = re.compile(r'^<2[a-z]{3}>\s*')
_W = re.compile(r"[a-zàèéìòùâêîôû']+")
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')], 'en': read(f'{DATA}/test.en')}
ban = set(test['scn']) | set(read(f'{DATA}/test.scn')) | set(test['en'])   # leakage guard, both stages

def sicilianish(t):
    """Italian leaks into the low-loss end: at most 1 word in -o, and no more than words in -u."""
    ws = [w for w in _W.findall(t.lower()) if len(w) >= 3]
    o = sum(w.endswith(('o', 'ò')) for w in ws)
    return o <= 1 and o <= sum(w.endswith(('u', 'ù')) for w in ws)

def load_scored(fname, src, tgt, k):
    # Column 1 is Sockeye's negative log probability that the target is a translation of
    # the source, so a low score is a good score (E. Wdowiak, 2026-09-16): exp(-score) is
    # the probability, 0.10 -> 90.5%, 1.00 -> 36.8%, 3.00 -> 5.0%. Take the k lowest, and
    # drop what scores low for the wrong reason: <unk> lines, short web fragments,
    # near-copies of the source, and Italian passing as Sicilian.
    rows = []
    for line in read(f'{BT}/{fname}'):
        if '<unk>' in line: continue
        p = line.split('\t')
        if len(p) < 3: continue
        try: sc = float(p[0])
        except ValueError: continue
        s = _TAG.sub('', p[1]).strip(); t = p[2].strip()
        if s and t and s != t and s not in ban and t not in ban: rows.append((sc, s, t))
    rows.sort(key=lambda r: r[0])
    kept = []
    for sc, s, t in rows:
        if len(s.split()) < MINW or len(t.split()) < MINW: continue
        if not sicilianish(t if tgt == 'scn' else s if src == 'scn' else 'u'): continue
        sm = SequenceMatcher(None, s.lower(), t.lower())
        if sm.quick_ratio() >= COPY and sm.ratio() >= COPY: continue
        kept.append((sc, s, t))
        if len(kept) == k: break
    cut = kept[-1][0] if kept else float('nan')
    print(f'  {fname}: kept {len(kept)} ({src}->{tgt}), '
          f'worst kept score {cut:.3f} = p {100 * math.exp(-cut):.1f}%')
    return ([r[1] for r in kept], [r[2] for r in kept], src, tgt)

stage2 = [
    load_scored('nllb-as-mono-eng_bt-s2e.scores.txt', 'scn', 'en', TOPK),
    load_scored('nllb-as-mono-scn_bt-e2s.scores.txt', 'en', 'scn', TOPK),
    load_scored('nllb-as-mono-scn_bt-i2s.scores.txt', 'it', 'scn', TOPK),
    load_scored('wm-million-as-mono-ita_bt-s2i.scores.txt', 'scn', 'it', TOPK),
    load_scored('as-mono-scn_bt-e2s.scores.txt', 'en', 'scn', TOPK),
    load_scored('as-mono-scn_bt-i2s.scores.txt', 'it', 'scn', TOPK),
]
wm = [(e, i) for e, i in zip(read(f'{BT}/wm-million_tkn2ascii.en-it.txt.en'),
                              read(f'{BT}/wm-million_tkn2ascii.en-it.txt.it')) if e not in ban][:TOPK]
wm_en = [e for e, _ in wm]; wm_it = [i for _, i in wm]
stage2 += [(wm_en, wm_it, 'en', 'it'), (wm_it, wm_en, 'it', 'en')]

m, tok = load_base('facebook/nllb-200-1.3B'); ad = attach_lora(m)
ad = finetune(ad, tok, build_dataset(tok, stage2),
              out_dir=f'{OUT}/nllb-lora-stage2-1.3B', epochs=1, batch_size=16, grad_accum=1)

# stage 3: combined high-quality fine-tune (leakage-guarded, std-space eval)
tr = {'scn': read(f'{FT}/train.scn-en.scn'), 'en': read(f'{FT}/train.scn-en.en')}
ti = {'scn': read(f'{FT}/train.scn-it.scn'), 'it': read(f'{FT}/train.scn-it.it')}
se = [(s, e) for s, e in zip(tr['scn'], tr['en']) if s not in ban and e not in ban]
ds3 = build_dataset(tok, [([s for s, _ in se], [e for _, e in se], 'scn', 'en'),
                          ([e for _, e in se], [s for s, _ in se], 'en', 'scn'),
                          (ti['scn'], ti['it'], 'scn', 'it'),
                          (ti['it'], ti['scn'], 'it', 'scn')])
ad = finetune(ad, tok, ds3, out_dir=f'{OUT}/nllb-lora-full-1.3B', epochs=2, batch_size=16, grad_accum=1)

res = {}
for s, t in [('scn', 'en'), ('en', 'scn')]:
    res[f'{s}->{t}'] = score(translate(ad, tok, test[s], s, t), test[t])
    print(f'{s}->{t}:  BLEU={res[f"{s}->{t}"][0]}  chrF={res[f"{s}->{t}"][1]}')
json.dump(res, open(f'{OUT}/results_full.json', 'w'), indent=2)

### 7.9 Curriculum: use all the data, worst first (Colab Pro)
Eryk's proposal (2026-09-18): rather than discarding the back-translations that score badly,
spend them in a first, coarse stage, *"kind of like a fat hammer, bludgeoning and spreading
the output probabilities across dialects, leaving dull peaks in approximately the right
locations"*, and let the later stages sharpen those peaks, first with the good
back-translations (§7.8 stage 2), then with the curated set (§7.8 stage 3). Same data, same
test set and same stages as §7.8, with one stage prepended, so the two runs are comparable:
if the curriculum helps, `results_curriculum.json` beats `results_full.json`.

The coarse stage takes the WORST-scoring pairs per direction (`KBAD`), keeping the Italian
and other-dialect text that §7.8 filters out on purpose, since that spread is the point here;
`<unk>` lines, near-copies and anything touching the test set stay out of both stages. **Run
§7.8 first**: without its numbers this cell measures nothing.

In [ ]:
# --- 7.9 curriculum: coarse stage on the badly scored back-translations, then 7.8's stages ---
import math
import re
from difflib import SequenceMatcher
from normalize_scn import normalize
BT = f'{OUT}/eryk'; FT = f'{OUT}/finetune'
KBAD, KGOOD = 200000, 100000       # per-direction caps: coarse stage, then the good stage
COPY = 0.9                         # drop near-copies in both stages (they teach copying)
MINW, MINW_BAD = 5, 3              # the coarse stage may be noisy, but not menus and titles
_TAG = re.compile(r'^<2[a-z]{3}>\s*')
_W = re.compile(r"[a-zàèéìòùâêîôû']+")
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')], 'en': read(f'{DATA}/test.en')}
ban = set(test['scn']) | set(read(f'{DATA}/test.scn')) | set(test['en'])
FILES = [('nllb-as-mono-eng_bt-s2e.scores.txt', 'scn', 'en'),
         ('nllb-as-mono-scn_bt-e2s.scores.txt', 'en', 'scn'),
         ('nllb-as-mono-scn_bt-i2s.scores.txt', 'it', 'scn'),
         ('wm-million-as-mono-ita_bt-s2i.scores.txt', 'scn', 'it'),
         ('as-mono-scn_bt-e2s.scores.txt', 'en', 'scn'),
         ('as-mono-scn_bt-i2s.scores.txt', 'it', 'scn')]

def sicilianish(t):
    """Italian leaks into the low-score end: at most 1 word in -o, and no more than words in -u."""
    ws = [w for w in _W.findall(t.lower()) if len(w) >= 3]
    o = sum(w.endswith(('o', 'ò')) for w in ws)
    return o <= 1 and o <= sum(w.endswith(('u', 'ù')) for w in ws)

def load_band(fname, src, tgt, k, coarse):
    """k pairs from one score file: the worst-scoring ones if `coarse`, else §7.8's selection."""
    rows = []
    for line in read(f'{BT}/{fname}'):
        if '<unk>' in line: continue
        p = line.split('\t')
        if len(p) < 3: continue
        try: sc = float(p[0])
        except ValueError: continue
        s = _TAG.sub('', p[1]).strip(); t = p[2].strip()
        if not s or not t or s == t or s in ban or t in ban: continue
        if len(s.split()) < (MINW_BAD if coarse else MINW): continue
        if len(t.split()) < (MINW_BAD if coarse else MINW): continue
        if not coarse and not sicilianish(t if tgt == 'scn' else s if src == 'scn' else 'u'): continue
        rows.append((sc, s, t))
    rows.sort(key=lambda r: r[0], reverse=coarse)
    kept = []
    for r in rows:
        sm = SequenceMatcher(None, r[1].lower(), r[2].lower())
        if sm.quick_ratio() < COPY or sm.ratio() < COPY:
            kept.append(r)
            if len(kept) == k: break
    lo, hi = kept[0][0], kept[-1][0]
    print(f'  {fname}: {len(kept)} pairs ({src}->{tgt}), scores {lo:.2f} to {hi:.2f} '
          f'= p {100 * math.exp(-lo):.1f}% to {100 * math.exp(-hi):.1f}%')
    return ([r[1] for r in kept], [r[2] for r in kept], src, tgt)

print('coarse stage (worst scores):')
stage_bad = [load_band(f, s, t, KBAD, True) for f, s, t in FILES]
print('good stage (best scores):')
stage_good = [load_band(f, s, t, KGOOD, False) for f, s, t in FILES]
wm = [(e, i) for e, i in zip(read(f'{BT}/wm-million_tkn2ascii.en-it.txt.en'),
                             read(f'{BT}/wm-million_tkn2ascii.en-it.txt.it')) if e not in ban][:KGOOD]
wm_en = [e for e, _ in wm]; wm_it = [i for _, i in wm]
stage_good += [(wm_en, wm_it, 'en', 'it'), (wm_it, wm_en, 'it', 'en')]

m, tok = load_base('facebook/nllb-200-1.3B'); ad = attach_lora(m)
ad = finetune(ad, tok, build_dataset(tok, stage_bad),
              out_dir=f'{OUT}/nllb-lora-coarse-1.3B', epochs=1, batch_size=16, grad_accum=1)
ad = finetune(ad, tok, build_dataset(tok, stage_good),
              out_dir=f'{OUT}/nllb-lora-cur-stage2-1.3B', epochs=1, batch_size=16, grad_accum=1)

# final stage: the curated set, exactly as in 7.8 (leakage-guarded, std-space eval)
tr = {'scn': read(f'{FT}/train.scn-en.scn'), 'en': read(f'{FT}/train.scn-en.en')}
ti = {'scn': read(f'{FT}/train.scn-it.scn'), 'it': read(f'{FT}/train.scn-it.it')}
se = [(s, e) for s, e in zip(tr['scn'], tr['en']) if s not in ban and e not in ban]
ds3 = build_dataset(tok, [([s for s, _ in se], [e for _, e in se], 'scn', 'en'),
                          ([e for _, e in se], [s for s, _ in se], 'en', 'scn'),
                          (ti['scn'], ti['it'], 'scn', 'it'),
                          (ti['it'], ti['scn'], 'it', 'scn')])
ad = finetune(ad, tok, ds3, out_dir=f'{OUT}/nllb-lora-curriculum-1.3B',
              epochs=2, batch_size=16, grad_accum=1)

res = {}
for s, t in [('scn', 'en'), ('en', 'scn')]:
    res[f'{s}->{t}'] = score(translate(ad, tok, test[s], s, t), test[t])
json.dump(res, open(f'{OUT}/results_curriculum.json', 'w'), indent=2)
try: base = json.load(open(f'{OUT}/results_full.json'))
except FileNotFoundError: base = {}
for k, v in res.items():
    b = base.get(k)
    delta = f'  (7.8: BLEU={b[0]} chrF={b[1]}, {v[0] - b[0]:+.2f} BLEU)' if b else ''
    print(f'{k}:  BLEU={v[0]}  chrF={v[1]}{delta}')

coarse stage (worst scores):
  nllb-as-mono-eng_bt-s2e.scores.txt: 200000 pairs (scn->en), scores 7.65 to 1.48 = p 0.0% to 22.9%
  nllb-as-mono-scn_bt-e2s.scores.txt: 200000 pairs (en->scn), scores 10.79 to 3.55 = p 0.0% to 2.9%
  nllb-as-mono-scn_bt-i2s.scores.txt: 200000 pairs (it->scn), scores 9.08 to 2.18 = p 0.0% to 11.4%
  wm-million-as-mono-ita_bt-s2i.scores.txt: 200000 pairs (scn->it), scores 5.80 to 0.52 = p 0.3% to 59.7%
  as-mono-scn_bt-e2s.scores.txt: 13815 pairs (en->scn), scores 6.98 to 0.07 = p 0.1% to 93.5%
  as-mono-scn_bt-i2s.scores.txt: 12291 pairs (it->scn), scores 3.55 to 0.06 = p 2.9% to 94.3%
good stage (best scores):
  nllb-as-mono-eng_bt-s2e.scores.txt: 100000 pairs (scn->en), scores 0.06 to 0.36 = p 94.0% to 69.6%
  nllb-as-mono-scn_bt-e2s.scores.txt: 100000 pairs (en->scn), scores 0.05 to 0.96 = p 95.5% to 38.3%
  nllb-as-mono-scn_bt-i2s.scores.txt: 100000 pairs (it->scn), scores 0.06 to 0.54 = p 94.6% to 58.0%
  wm-million-as-mono-ita_bt-s2i.scores.txt: 1000

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/13815 [00:00<?, ? examples/s]

Map:   0%|          | 0/12291 [00:00<?, ? examples/s]

Step,Training Loss
100,1.729044
200,1.509460
300,1.519080
400,1.434340
500,1.440104
600,1.402166
700,1.423238
800,1.370812
900,1.337578
1000,1.390334


Step,Training Loss
100,1.729044
200,1.509460
300,1.519080
400,1.434340
500,1.440104
600,1.402166
700,1.423238
800,1.370812
900,1.337578
1000,1.390334


## 8. Results & discussion

On our frozen literary test set (scn→en): floor 5.27 → Sockeye levers 10.85 → NLLB-1.3B
zero-shot 29.0 → **bidirectional LoRA 31.4** → + back-translation **31.6**. The reverse
direction en→scn rises 9.9 → **18.7**; Italian is strong (scn→it **43.5**, it→scn **27.0**).

**Method vs data.** Each feasible lever adds little — multilingual ≈0 on scn↔en, normalization
+0.4, back-translation +0.17 (capped by our 7.6k monolingual pool vs his ~550k). So the
**method** gap to Wdowiak's reverse-training state of the art (Sc→En 48.6, En→Sc 45.1, It↔Sc
61–63, on his in-domain test set) is largely closed; the residual is **data** — his
hand-curated corpus, far more back-translation, and an in-domain test set. This is a concrete
argument for combining his corpus with our pipeline rather than competing.

## 9. Serving
The saved adapter backs a local web app and a Telegram bot — one FastAPI `/translate`
endpoint over `experiments/serving/translator.py`. See `experiments/serving/README.md`:
`uvicorn api:app` serves the site at `/`, `telegram_bot.py` runs the bot, and a Docker +
compose deploy scaffold is in the same folder.

## 10. Conclusion
A fully reproducible, Perl-free path from PDFs to a trilingual Sicilian translator reaches
31.6 BLEU (scn→en) — above the published baseline, below the reverse-training SOTA. By
matching the feasible method levers and measuring their small effect, we localize the
remaining gap to **data**, motivating collaboration with the original author.